# Adjusted trial time boundaries
This notebook creates the `ANIMAL_TRIAL_TIME_BOUNDARIES` dictionary used in the Day 2 and 3 behavioral analyseses.

The input is the Excel file produced by `export_adjusted_fps_only()`, which is a function inside `create_frame_based_eopchs (OR OTHER NAME CHECK`. That file contains the adjusted FPS and the first/last LED onset frames for each subject. These values are converted to trial start and end times in seconds. 

## 0. Import functions and libraries

In [1]:
import pickle
import os
import pandas as pd
import math
import matplotlib.pyplot as plt
import numpy as np

from mrondes_files.helper_functions import select_file

## 1. Select adjusted fps summary excel sheet

In [2]:
INPUT_FILE = select_file("select file that holds the adjusted fps for each animal")

### 1.2. (Optional) Define output file path

In [3]:
OUTPUT_DIR = os.path.dirname(INPUT_FILE)
print(OUTPUT_DIR)

OUTPUT_TXT_FILE = os.path.join(OUTPUT_DIR, "animal_trial_time_boundaries_day2.txt") # Or day 3

/Users/febevansommeren/Documents/EEG


## 2. Create helper function 

In [4]:
def create_animal_trial_time_boundaries(
    fps_summary_file,
    sheet_name=0,
    round_times=True,
    save_txt_file=None,
):
    """
    Create ANIMAL_TRIAL_TIME_BOUNDARIES from an adjusted FPS summary Excel file.
    This is used for behavioral analyses. 

    The Excel file is expected to contain the columns produced by
    export_adjusted_fps_only(): batch_cage, adjusted_fps,
    first_led_onset_frame, and last_led_onset_frame.

    Parameters:
    ### fps_summary_file [str]: Path to the adjusted FPS summary Excel file.
    ### sheet_name: Sheet to read from the Excel file. Default is the first sheet.
    ### round_times [bool, optional]: If True, start and end times are rounded to integers. If False,
        times are kept as floats. Default is True.
    ### save_txt_file [str | None]: Optional path to save the dictionary as a text file.

    Returns:
    ### animal_trial_time_boundaries, boundary_summary_df [tuple[dict, pandas.DataFrame]]:
        Dictionary with animal IDs as keys and (start_time, end_time) tuples
        as values, plus a summary dataframe used to create the dictionary.
    """
    # Read excel sheet
    fps_df = pd.read_excel(fps_summary_file, sheet_name=sheet_name)
    fps_df.columns = fps_df.columns.str.strip()

    required_cols = [
        "batch_cage",
        "adjusted_fps",
        "first_led_onset_frame",
        "last_led_onset_frame",
    ]

    missing_cols = [col for col in required_cols if col not in fps_df.columns]
    if missing_cols:
        raise ValueError(
            "The adjusted FPS file is missing required columns: "
            + ", ".join(missing_cols)
        )

    summary_rows = []

    # Iterate over all animals in the excel file 
    for _, row in fps_df.iterrows():
        batch_cage = str(row["batch_cage"]).strip()
        adjusted_fps = pd.to_numeric(row["adjusted_fps"], errors="coerce")
        first_frame = pd.to_numeric(row["first_led_onset_frame"], errors="coerce")
        last_frame = pd.to_numeric(row["last_led_onset_frame"], errors="coerce")

        if pd.isna(adjusted_fps) or adjusted_fps <= 0:
            print(f"{batch_cage} - invalid adjusted FPS, using 30 FPS instead")
            adjusted_fps = 30

        if pd.isna(first_frame) or pd.isna(last_frame):
            print(f"Skipping {batch_cage} - missing LED onset frame")
            continue

        # Extract animal ID from the end of batch_cage, e.g. b2c3.2_8991 -> 8991
        animal_id = batch_cage.rsplit("_", 1)[-1]

        # Get start and end of trial times
        first_time = first_frame / adjusted_fps
        last_time = last_frame / adjusted_fps

        # Round up to integers
        if round_times:
            first_time = int(round(first_time))
            last_time = int(round(last_time))

        summary_rows.append({
            "animal_id": animal_id,
            "batch_cage": batch_cage,
            "adjusted_fps": adjusted_fps,
            "first_led_onset_frame": first_frame,
            "last_led_onset_frame": last_frame,
            "trial_start_s": first_time,
            "trial_end_s": last_time,
        })

    boundary_summary_df = pd.DataFrame(summary_rows)

    duplicate_ids = boundary_summary_df[
        boundary_summary_df["animal_id"].duplicated(keep=False)
    ]["animal_id"].unique()

    if len(duplicate_ids) > 0:
        raise ValueError(
            "Duplicate animal IDs found. Check batch_cage values for: "
            + ", ".join(map(str, duplicate_ids))
        )

    animal_trial_time_boundaries = {
        str(row["animal_id"]): (row["trial_start_s"], row["trial_end_s"])
        for _, row in boundary_summary_df.iterrows()
    }

    if save_txt_file is not None:
        with open(save_txt_file, "w") as f:
            f.write("ANIMAL_TRIAL_TIME_BOUNDARIES = {\n")
            for animal_id, times in animal_trial_time_boundaries.items():
                f.write(f"    {animal_id!r}: {times},\n")
            f.write("}\n")

    return animal_trial_time_boundaries, boundary_summary_df

## 4. Generate `ANIMAL_TRIAL_TIME_BOUNDARIES`

In [6]:
ANIMAL_TRIAL_TIME_BOUNDARIES, trial_boundaries_df = create_animal_trial_time_boundaries(
    INPUT_FILE,
    round_times=True,
    save_txt_file=OUTPUT_TXT_FILE,
)

trial_boundaries_df.head()

,animal_id,batch_cage,adjusted_fps,first_led_onset_frame,last_led_onset_frame,trial_start_s,trial_end_s
0,8989,b1c1.1_8989,30.230981,145,18168,5,601
1,8993,b1c1.2_8993,29.972645,111,18135,4,605
2,8982,b1c1.3_8982,29.977075,65,19578,2,653
3,8995,b1c1.4_8995,29.977332,100,18124,3,605
4,8990,b1c2.1_8990,29.974859,53,19565,2,653


## 5. Print the dictionary

This prints the dictionary in a format that can be copied directly into the behavioral analysis notebook.

In [7]:
print("ANIMAL_TRIAL_TIME_BOUNDARIES = {")
for animal_id, boundaries in ANIMAL_TRIAL_TIME_BOUNDARIES.items():
    print(f"    {animal_id!r}: {boundaries},")
print("}")

print(f"\nSaved dictionary text file to: {OUTPUT_TXT_FILE}")

ANIMAL_TRIAL_TIME_BOUNDARIES = {
    '8989': (5, 601),
    '8993': (4, 605),
    '8982': (2, 653),
    '8995': (3, 605),
    '8990': (2, 653),
    '8903': (5, 606),
    '8998': (6, 657),
    '8981': (7, 608),
    '8987': (3, 604),
    '8996': (3, 604),
    '8994': (4, 605),
    '8992': (4, 605),
    '8985': (4, 605),
    '8997': (4, 606),
    '9000': (5, 606),
    '9001': (3, 604),
    '9007': (4, 605),
    '9002': (5, 606),
    '9005': (6, 607),
    '9012': (3, 604),
    '9010': (3, 605),
    '9020': (3, 604),
    '9003': (3, 605),
    '8988': (3, 537),
    '9013': (5, 606),
    '9017': (3, 604),
    '9009': (1, 602),
    '9008': (1, 602),
    '9004': (0, 601),
    '9018': (1, 605),
    '9014': (3, 604),
    '9016': (1, 604),
    '9011': (3, 604),
    '8858': (2, 622),
    '8760': (2, 603),
    '8851': (3, 604),
    '8859': (2, 607),
    '8857': (1, 602),
    '8854': (1, 602),
    '8855': (1, 602),
    '9015': (3, 604),
}

Saved dictionary text file to: /Users/febevansommeren/Document

## 6. Add missing values 

The trial time boundaries used for the behavioral analyseses were primarily based on the adjusted FPS values calculated from the synchronization between EEG TTL events and LED onset frames. These adjusted FPS values were used to convert video frame numbers into trial times in seconds for each animal.

For most animals, the start and end times were obtained from the adjusted FPS-based calculation. However, for a few animals, adjusted FPS-based trial boundaries were missing. In those cases, the provided trial boundaries (Provided in code of M. Ronde) were used instead. 

This ensures that all included animals have defined trial start and end times, while still prioritizing the adjusted FPS-based timing information whenever available.

In [8]:
# Day 2
animal_trial_time_boundaries = {
    # Batch 1
    '8989': (5, 606),
    '8993': (4, 605),
    '8982': (2, 653),
    '8995': (3, 605),
    '8990': (2, 653),
    '8903': (5, 606), 
    '8998': (6, 657),
    '8981': (7, 608),
    '8987': (3, 604),
    '8988': (3, 604),
    '8996': (3, 604),
    '8994': (4, 605),
    '8983': (3, 605),
    '8992': (4, 605),
    '8985': (4, 604),
    '8997': (4, 606),
    '9000': (5, 606),
    # Batch 2
    '9001': (3, 604),
    '9007': (4, 605),
    '9002': (5, 606),
    '9005': (5, 607),
    '9012': (3, 604),
    '9010': (3, 605),
    '9020': (3, 604),
    '9003': (3, 605),
    '8991': (5, 606),
    '8986': (3, 604),
    #'8999': (4, 605), # sick animal, not used
    '9013': (5, 606),
    '9017': (3, 604), 
    '9019': (3, 604),
    # Batch 3
    '9009': (1, 602),
    '9008': (1, 602),
    '9004': (0, 601),
    '9018': (1, 602),
    '9014': (3, 604),
    '9016': (1, 602),
    '9011': (3, 604),
    '8858': (2, 622),
    '8760': (2, 603),
    '8851': (3, 604),
    '8859': (2, 607),
    '8857': (1, 602),
    '8854': (1, 602),
    '8855': (1, 602),
    '9015': (3, 604),
} 

# Day 3
animal_trial_time_boundaries = {
    # Batch 1
    '8989': (4, 605),
    '8993': (7, 609),
    '8982': (21, 625),
    '8995': (10, 611),
    '8990': (4, 472), # 2 videos available, 11_27_13 used
    '8903': (4, 606),
    '8998': (1, 717),
    '8981': (4, 605),
    '8987': (4, 605),
    '8988': (2, 604),
    '8996': (6, 607),
    '8994': (5, 606),
    '8983': (5, 606),
    '8992': (3, 604),
    '8985': (3, 604),
    '8997': (3, 604),
    '9000': (4, 605), 
    # Batch 2
    '9001': (1, 602),
    '9007': (0, 602),
    '9002': (3, 604),
    '9005': (3, 604),
    '9012': (2, 604),
    '9010': (5, 603),
    '9020': (6, 607),
    '9003': (5, 606),
    '8991': (3, 604),
    '8986': (2, 603),
    #'8999': #b2c3F3tg 4 animal 8999, day 3 excluded
    '9013': (3, 604),
    '9017': (1, 603),
    '9019': (3, 604),
    # Batch 3
    '9009': (1, 602),
    '9008': (1, 602),
    '9018': (4, 609),
    '9004': (0, 601),
    '9014': (4, 605),
    '9016': (1, 602),
    '9011': (3, 604),
    '8858': (1, 602),
    '8760': (1, 606),
    '8851': (1, 602),
    '8859': (1, 602),
    '8857': (1, 602),
    '8854': (1, 602),
    '8855': (1, 602),
    '9015': (1, 603),
}